# S03 — Sync Amazon S3 to an Ubuntu Local Directory

This notebook performs the reverse of S02. It downloads the contents of an existing S3 bucket in `us-east-1` to `~/datalake_from_s3` on Ubuntu. It uses the AWS CLI profile named `training`.

The bucket name is read from the `S3_BUCKET_NAME` environment variable. If that variable is not set, the notebook uses `gksdatalake`.

## 1. Prerequisites

Before continuing:

- AWS CLI v2 must be installed.
- The `training` profile must be configured.
- The source S3 bucket must already exist.
- The profile must have permission to list and download objects from the bucket.
- The Ubuntu user must have permission to write in the home directory.

In [ ]:
%%bash
set -euo pipefail
aws --version
aws sts get-caller-identity --profile training
aws configure get region --profile training

## 2. Set or inspect the source and destination

To use a bucket other than `gksdatalake`, set the environment variable before starting Jupyter:

```bash
export S3_BUCKET_NAME=my-existing-bucket
jupyter lab
```

The local destination is fixed as `~/datalake_from_s3`.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
LOCAL_BACKUP="$HOME/datalake_from_s3"
printf 'S3 source: s3://%s/\nLocal destination: %s\n' "$S3_BUCKET_NAME" "$LOCAL_BACKUP"

## 3. Verify access to the S3 bucket

This read-only check confirms that the bucket is accessible before starting the backup.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
aws s3api head-bucket --bucket "$S3_BUCKET_NAME" --profile training
aws s3 ls "s3://$S3_BUCKET_NAME/" \
  --recursive \
  --human-readable \
  --summarize \
  --region us-east-1 \
  --profile training

## 4. Preview the S3-to-local sync

Run with `--dryrun` first. It shows which objects would be downloaded, which local files would be overwritten, and which extra local files would be deleted without making any changes. With `--exact-timestamps`, a same-size local file is skipped only when its timestamp also matches the S3 object.

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
LOCAL_BACKUP="$HOME/datalake_from_s3"
aws s3 sync "s3://$S3_BUCKET_NAME/" "$LOCAL_BACKUP/" \
  --dryrun \
  --exact-timestamps \
  --delete \
  --region us-east-1 \
  --profile training

## 5. Sync S3 to `~/datalake_from_s3`

Run this cell only after reviewing the dry-run output. AWS CLI creates `~/datalake_from_s3` if it does not exist, downloads new objects, and overwrites differing local files. Because `--delete` is enabled, files inside `~/datalake_from_s3` that do not exist in the S3 bucket are deleted.

adding option delete below would delete files which are present in local but not present in s3 bucket.

`--delete \`

In [ ]:
%%bash
set -euo pipefail
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
LOCAL_BACKUP="$HOME/datalake_from_s3"
aws s3 sync "s3://$S3_BUCKET_NAME/" "$LOCAL_BACKUP/" \
  --exact-timestamps \
  --region us-east-1 \
  --profile training

## 6. Verify the local backup

Confirm that the destination exists, list the first 20 downloaded files, and display its total disk usage.

In [ ]:
%%bash
set -euo pipefail
LOCAL_BACKUP="$HOME/datalake_from_s3"

if [[ ! -d "$LOCAL_BACKUP" ]]; then
  echo "Backup directory not found: $LOCAL_BACKUP" >&2
  exit 1
fi

find "$LOCAL_BACKUP" -type f | sed -n '1,20p'
du -sh "$LOCAL_BACKUP"

## Sync command summary

The essential Ubuntu command is:

```bash
S3_BUCKET_NAME="${S3_BUCKET_NAME:-gksdatalake}"
aws s3 sync "s3://$S3_BUCKET_NAME/" "$HOME/datalake_from_s3/" --exact-timestamps --delete --region us-east-1 --profile training
```